# FashionCLIP Equivalence Check

**Purpose:** Verify the team classifier's split path (cached text features, then image
features and a dot product) produces the same logits as the reference FashionCLIP
joint forward pass, on the checkpoint the pipeline uses.  
**Inputs:** the `patrickjohncyh/fashion-clip` checkpoint; a random image fixed by
seed 0.  
**Outputs:** printed logits from both paths and an allclose verdict at atol 1e-4; no
files are written.  
**Backs:** no results/ artefact; it underwrites
`basketball/team_classifier/classifier.py`, which classifies with exactly this split
path.

The TqdmWarning and tokenizer FutureWarning in the output are environment notices and
harmless.

In [1]:
import torch, numpy as np
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

m = CLIPModel.from_pretrained('patrickjohncyh/fashion-clip').eval()
p = CLIPProcessor.from_pretrained('patrickjohncyh/fashion-clip')
classes = ['white basketball jersey', 'dark blue basketball jersey']

rng = np.random.default_rng(0)
img = Image.fromarray(rng.integers(0, 255, (224, 224, 3), dtype=np.uint8))

with torch.inference_mode():
    ref = m(**p(text=classes, images=img, return_tensors='pt', padding=True)).logits_per_image

    tf = m.get_text_features(**p.tokenizer(classes, return_tensors='pt', padding=True))
    tf = tf / tf.norm(dim=-1, keepdim=True)
    imf = m.get_image_features(**p.image_processor(img, return_tensors='pt'))
    imf = imf / imf.norm(dim=-1, keepdim=True)
    fast = m.logit_scale.exp() * imf @ tf.T

print('reference logits:', ref)
print('fast-path logits:', fast)
print('allclose:', torch.allclose(ref, fast, atol=1e-4))

/home/jovyan/basketball_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


reference logits: tensor([[12.2493,  9.8841]])
fast-path logits: tensor([[12.2493,  9.8841]])
allclose: True


The two paths agree within 1e-4 (allclose True in the output above), so results
computed through the classifier's split path stand for the reference model's.